# Deep-learning progression models: feature-recipe comparison

This notebook evaluates **FusionMLP** and **PairModel** with the same two fold-local feature recipes used by the linear models:

- **FRDA-only:** selected from outer-training FRDA progression.
- **Control-aware:** selected from outer-training FRDA and controls.

Both neural models are trained only on FRDA. The fitted FRDA scaler and network are then applied unchanged to held-out FRDA and held-out controls. Clinical-score heads are disabled.

## 1. Shared data, folds, and feature recipes

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
import torch


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find repository root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import set_global_seeds
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.control_aware_selection import wide_cohort_to_pair_long
from src.eval.recipe_dl import run_dl_recipe_comparison
from src.features.panels import a_priori_70_feature_names
from src.reporting.experiment_artifacts import (
    read_experiment_contract,
    read_table_artifact,
    write_table_artifact,
)

RUN_ID = "trackfa_70_feature_comparison_v1"
RUN_DIR = REPO_ROOT / "results" / "experiments" / RUN_ID
SELECTION_DIR = RUN_DIR / "selections"
MODEL_DIR = RUN_DIR / "models" / "deep_learning"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

manifest, folds = read_experiment_contract(RUN_DIR)
feature_panel = a_priori_70_feature_names()
recipes = pd.concat([
    read_table_artifact(SELECTION_DIR / "frda_only_features_by_fold.csv", schema="feature_recipe", manifest=manifest, panel=feature_panel),
    read_table_artifact(SELECTION_DIR / "control_aware_features_by_fold.csv", schema="feature_recipe", manifest=manifest, panel=feature_panel),
], ignore_index=True)

pairs = pd.read_csv(manifest["data_path"])
frda_long = trackfa_pairs_to_long(pairs)
wide = pd.read_csv(Path(manifest["data_path"]).with_name("trackfa_merged_wide.csv"), low_memory=False)
control_long = wide_cohort_to_pair_long(wide, feature_panel, cohort_value=1)
groups = infer_trackfa_feature_groups(pairs)
structural_features = list(groups.poms + groups.brainspinemorph)
diffusion_features = list(groups.braindti)

set_global_seeds(int(manifest["seed"]))
DEVICE = torch.device("cpu")
display(pd.DataFrame([
    {"Cohort": "FRDA", "Participants": frda_long["subject"].nunique(), "Annual pairs": frda_long["pair_id"].nunique()},
    {"Cohort": "Control", "Participants": control_long["subject"].nunique(), "Annual pairs": control_long["pair_id"].nunique()},
]))
display(pd.DataFrame([
    {"Feature recipe": "FRDA-only", "Selection data": "Outer-training FRDA", "Model training": "FRDA only"},
    {"Feature recipe": "Control-aware", "Selection data": "Outer-training FRDA + controls", "Model training": "FRDA only"},
]))
print("Device:", DEVICE, "| run:", RUN_ID)

,Cohort,Participants,Annual pairs
0,FRDA,117,207
1,Control,95,190


,Feature recipe,Selection data,Model training
0,FRDA-only,Outer-training FRDA,FRDA only
1,Control-aware,Outer-training FRDA + controls,FRDA only


Device: cpu | run: trackfa_70_feature_comparison_v1


## 2. Training policy

Early stopping uses participants drawn only from each outer-training FRDA fold. The outer-test folds are used once for performance estimation. The CPU backend is used here for reproducibility.

In [ ]:
FUSION_SETTINGS = {
    "epochs": 60, "patience": 8, "lr": 3e-3, "weight_decay": 1e-5,
    "dropout": 0.0, "z_clip": 3.0, "val_fraction": 0.2,
    "use_clinical_heads": False, "lambda_prog": 1.0, "lambda_fars": 0.0, "lambda_sara": 0.0,
}
PAIR_SETTINGS = {
    "epochs": 60, "patience": 8, "lr": 3e-3, "weight_decay": 1e-5,
    "dropout": 0.0, "z_clip": None, "val_fraction": 0.2,
    "use_clinical_heads": False, "lambda_prog": 1.0, "lambda_fars": 0.0, "lambda_sara": 0.0,
}
assert not FUSION_SETTINGS["use_clinical_heads"] and not PAIR_SETTINGS["use_clinical_heads"]
assert FUSION_SETTINGS["lambda_fars"] == FUSION_SETTINGS["lambda_sara"] == 0
assert PAIR_SETTINGS["lambda_fars"] == PAIR_SETTINGS["lambda_sara"] == 0

display(pd.DataFrame([
    {"Architecture": "FusionMLP", "Epoch limit": FUSION_SETTINGS["epochs"], "Patience": FUSION_SETTINGS["patience"], "Clinical heads": "Disabled"},
    {"Architecture": "PairModel", "Epoch limit": PAIR_SETTINGS["epochs"], "Patience": PAIR_SETTINGS["patience"], "Clinical heads": "Disabled"},
]))

,Architecture,Epoch limit,Patience,Clinical heads
0,FusionMLP,60,8,Disabled
1,PairModel,60,8,Disabled


: 

## 3. Held-out FRDA and control comparison

In [ ]:
result = run_dl_recipe_comparison(
    frda_long,
    control_long,
    folds,
    recipes,
    run_id=RUN_ID,
    structural_features=structural_features,
    diffusion_features=diffusion_features,
    device=DEVICE,
    fusion_kwargs=FUSION_SETTINGS,
    pair_kwargs=PAIR_SETTINGS,
    seed=int(manifest["seed"]),
    n_boot=500,
)

write_table_artifact(MODEL_DIR / "oof_visit_scores.csv", result["oof_visit_scores"], schema="oof_visit_scores", manifest=manifest)
write_table_artifact(MODEL_DIR / "performance.csv", result["performance"], schema="performance", manifest=manifest)
result["training_diagnostics"].to_csv(MODEL_DIR / "training_diagnostics.csv", index=False)
result["site_diagnostics"].to_csv(MODEL_DIR / "site_diagnostics.csv", index=False)
result["comparison"].to_csv(MODEL_DIR / "headline_comparison.csv", index=False)

columns = [
    "model", "selection_strategy",
    "frda_pooled_annual_d_z", "frda_pooled_annual_ci_low", "frda_pooled_annual_ci_high",
    "frda_pooled_annual_n_participants", "frda_pooled_annual_n_pairs",
    "control_pooled_annual_d_z", "control_pooled_annual_ci_low", "control_pooled_annual_ci_high",
    "control_pooled_annual_n_participants", "control_pooled_annual_n_pairs",
    "signed_frda_control_contrast", "absolute_control_d_z",
    "frda_v1_v2_d_z", "frda_v2_v3_d_z", "control_v1_v2_d_z", "control_v2_v3_d_z",
    "frda_interval_gap",
]
headline = result["comparison"][[column for column in columns if column in result["comparison"]]].rename(columns={
    "model": "Architecture", "selection_strategy": "Feature selection",
})
print("Deep-learning held-out comparison")
display(headline.round(3))

## 4. Training and site diagnostics

In [ ]:
diagnostics = result["training_diagnostics"].rename(columns={
    "model": "Architecture", "selection_strategy": "Feature selection", "outer_fold": "Fold",
    "feature_count": "Features", "train_frda_participants": "Train FRDA N",
    "train_frda_pairs": "Train FRDA pairs", "test_frda_participants": "Test FRDA N",
    "test_control_participants": "Test control N", "best_epoch": "Best epoch",
    "runtime_seconds": "Runtime (s)",
})
display(diagnostics[[
    "Architecture", "Feature selection", "Fold", "Features", "Train FRDA N",
    "Train FRDA pairs", "Test FRDA N", "Test control N", "Best epoch", "Runtime (s)",
]].round(3))

site = result["site_diagnostics"].rename(columns={
    "model": "Architecture", "selection_strategy": "Feature selection", "cohort": "Cohort",
    "site_r2_delta": "Site partial R2", "site_p_value": "Site p-value",
})
print("Association between site and held-out longitudinal score change")
display(site[[column for column in [
    "Architecture", "Feature selection", "Cohort", "n", "site_levels", "Site partial R2", "Site p-value",
] if column in site]].round(3))

,Architecture,Feature selection,Fold,Features,Train FRDA N,Train FRDA pairs,Test FRDA N,Test control N,Best epoch,Runtime (s)
0,fusion_mlp,frda_only,1,16,93,165,24,14,60,3.530
1,fusion_mlp,frda_only,2,16,93,163,24,13,15,0.134
2,fusion_mlp,frda_only,3,16,94,167,23,13,5,0.078
3,fusion_mlp,frda_only,4,16,94,166,23,14,5,0.079
4,fusion_mlp,frda_only,5,16,94,167,23,13,11,0.115
5,fusion_mlp,control_aware,1,16,93,165,24,14,44,0.294
6,fusion_mlp,control_aware,2,16,93,163,24,13,21,0.175
7,fusion_mlp,control_aware,3,16,94,167,23,13,1,0.061
8,fusion_mlp,control_aware,4,16,94,166,23,14,3,0.068
9,fusion_mlp,control_aware,5,16,94,167,23,13,23,0.184


Association between site and held-out longitudinal score change


,Architecture,Feature selection,Cohort,n,site_levels,Site partial R2,Site p-value
0,fusion_mlp,frda_only,FRDA,207,6,0.027,0.346
1,fusion_mlp,frda_only,Control,126,6,0.021,0.763
2,fusion_mlp,control_aware,FRDA,207,6,0.060,0.028
3,fusion_mlp,control_aware,Control,126,6,0.026,0.675
4,pair_model,frda_only,FRDA,207,6,0.004,0.972
5,pair_model,frda_only,Control,126,6,0.029,0.609
6,pair_model,control_aware,FRDA,207,6,0.025,0.395
7,pair_model,control_aware,Control,126,6,0.045,0.353


## Interpretation

The primary comparison is between feature recipes within each architecture. A lower absolute control effect and a larger signed FRDA-control contrast indicate better disease specificity, provided FRDA sensitivity and interval consistency are retained.

These neural models are exploratory because the cohort is small relative to model flexibility. The final comparator reports them beside the simpler linear models rather than selecting a winner inside this notebook. SHAP should be run only after a DL configuration is locked.

In [ ]:
print("Machine-readable artifacts:", MODEL_DIR)

Machine-readable artifacts: /Users/robertwang/Documents/New_project/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/models/deep_learning
